# Example 2: several inputs, one output

This is the shortest complete example: three scalar inputs, one output with
two elements, and no options at all in the model description. Use it as a
template for your own models.

If you have not read [Example 1](../example1/example1.ipynb) yet, start
there: it explains each step in more detail.

In [1]:
import json

import numpy as np
import onnx
import pandas as pd
import torch
from torch import nn

## The model

The model computes `[x1 + x2, x2 - x3]`.

In [2]:
class ExampleModel(nn.Module):

    def forward(self, x1, x2, x3):
        x4 = x1 + x2
        x5 = x2 - x3
        return torch.cat([x4, x5], dim=0)


model = ExampleModel()
model.eval()

x1, x2, x3 = torch.randn(1), torch.randn(1), torch.randn(1)
print(model(x1, x2, x3))

tensor([0.6923, 1.3863])


## Export to ONNX

Each input is a tensor of shape `(1,)`, and the node names are `x_1`, `x_2`,
`x_3` and `y`.

In [3]:
model_name = "example2"

torch.onnx.export(
    model,
    (x1, x2, x3),
    f"{model_name}.onnx",
    input_names=["x_1", "x_2", "x_3"],
    output_names=["y"],
    external_data=False,  # keep the whole model in a single file
    verbose=False,
)

onnx_model = onnx.load(f"{model_name}.onnx")
onnx.checker.check_model(onnx_model)

## The model description

Only the required keys are used here: the model `name`, a `description`, the
`FMIVersion`, and the `name` of each input and output node.

In [4]:
model_description = {
    "name": "example2",
    "description": "The model defines a simple example model with multiple inputs and one output.",
    "FMIVersion": "2.0",
    "inputs": [
        {"name": "x_1", "description": "A scalar input to the model."},
        {"name": "x_2", "description": "A scalar input to the model."},
        {"name": "x_3", "description": "A scalar input to the model."}
    ],
    "outputs": [
        {"name": "y", "description": "The output array."}
    ]
}

with open(f"{model_name}Description.json", "w", encoding="utf-8") as f:
    json.dump(model_description, f, indent=4)

## Build the FMU

In [5]:
from loguru import logger
from onnx2fmu.app import build

logger.disable("onnx2fmu")  # keep the notebook output short

build(
    model_path=f"{model_name}.onnx",
    model_description_path=f"{model_name}Description.json",
    target_folder="temp",
)

## Reference data

The FMU variables are named after the node and the element index. A node
`x_1` with one element becomes the variable `x_1_0`, and the output `y` with
two elements becomes `y_0` and `y_1`.

We generate 100 random samples, compute the expected output with the PyTorch
model, and save both to CSV files.

In [6]:
time_steps = 100
rng = np.random.default_rng(0)
data = rng.random((time_steps, 3))

with torch.no_grad():
    y = torch.stack([
        model(*[torch.tensor(data[i, k:k + 1]) for k in range(3)])
        for i in range(time_steps)
    ]).numpy()

time = pd.Index(np.arange(time_steps), name="time")

pd.DataFrame(data, columns=["x_1_0", "x_2_0", "x_3_0"], index=time).to_csv("Example2_in.csv")
pd.DataFrame(y, columns=["y_0", "y_1"], index=time).to_csv("Example2_ref.csv")

## Simulate the FMU

The row of the results at time `t + 1` holds the output for the inputs at
time `t`, so the first row is dropped before comparing.

In [7]:
from fmpy import simulate_fmu

signals = np.genfromtxt("Example2_in.csv", delimiter=",", names=True)

results = simulate_fmu(
    f"{model_name}.fmu",
    start_time=0,
    stop_time=time_steps,
    output_interval=1,
    step_size=1,
    input=signals,
    validate=True,
)

df_fmu = pd.DataFrame(results).set_index("time")
fmu_output = df_fmu.iloc[1:time_steps + 1].to_numpy()

print(f"largest difference between FMU and PyTorch: {np.abs(fmu_output - y).max():.2e}")

comparison = pd.DataFrame(
    np.hstack([y, fmu_output]),
    columns=["y_0 (PyTorch)", "y_1 (PyTorch)", "y_0 (FMU)", "y_1 (FMU)"],
    index=time,
)
comparison.head()

largest difference between FMU and PyTorch: 1.02e-07


,y_0 (PyTorch),y_1 (PyTorch),y_0 (FMU),y_1 (FMU)
time,,,,
0,0.906748,0.228813,0.906748,0.228813
1,0.829798,-0.099485,0.829798,-0.099485
2,1.336132,0.185872,1.336132,0.185872
3,1.750926,0.813115,1.750926,0.813115
4,0.890990,-0.696070,0.890990,-0.696070


## What next

- [Example 3](../example3/example3.ipynb) wraps a trained recurrent network.
- [Example 4](../example4/example4.ipynb) shows how to give a model a memory
  with feedback states.